# Preparación de los datos: gatos y perros

En este notebook se prepara el conjunto de datos de **Quick, Draw!** que utilizaremos para entrenar y evaluar los modelos de clasificación y clustering.

[link-de-los-datos](https://console.cloud.google.com/storage/browser/quickdraw_dataset/full/numpy_bitmap;tab=objects?prefix=&forceOnObjectsSortingFiltering=false)

[link-github](https://github.com/googlecreativelab/quickdraw-dataset)

El proceso consiste en:
1. Cargar los dibujos originales de gatos y perros.
2. Seleccionar una cantidad equilibrada de ejemplos de cada clase.
3. Reservar 10 imágenes exclusivamente para una prueba individual posterior.
4. Dividir el resto de los datos en entrenamiento, validación y test.
5. Normalizar los valores de los píxeles entre 0 y 1.
6. Guardar los conjuntos preparados para utilizarlos posteriormente en los modelos.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

In [2]:
N_PER_CLASS = 10_000
RANDOM_STATE = 42

In [3]:
cat_path = Path("datos/full_numpy_bitmap_cat.npy")
dog_path = Path("datos/full_numpy_bitmap_dog.npy")

cats = np.load(cat_path)
dogs = np.load(dog_path)

print(f"Gatos disponibles: {len(cats):,}")
print(f"Perros disponibles: {len(dogs):,}")
print(f"Forma de cada dibujo: {cats.shape[1:]}")

Gatos disponibles: 123,202
Perros disponibles: 152,159
Forma de cada dibujo: (784,)


In [4]:
rng = np.random.default_rng(RANDOM_STATE)

cat_idx = rng.choice(len(cats), N_PER_CLASS, replace=False)
dog_idx = rng.choice(len(dogs), N_PER_CLASS, replace=False)

cats = cats[cat_idx]
dogs = dogs[dog_idx]

X = np.concatenate([cats, dogs], axis=0)

y = np.concatenate([
    np.zeros(N_PER_CLASS, dtype=np.uint8),
    np.ones(N_PER_CLASS, dtype=np.uint8)
])

## Reserva de imágenes para la prueba individual

Antes de realizar cualquier división de los datos, se reservan **10 imágenes** que no participarán en entrenamiento, validación ni test.

Estas imágenes se utilizarán posteriormente para comparar, de forma individual, la etiqueta asignada y el tiempo de predicción de cada modelo.

In [5]:
rng = np.random.default_rng(42)

indices_sobrantes = rng.choice(
    len(X),
    size=10,
    replace=False
)

X_sobrante = X[indices_sobrantes]
y_sobrante = y[indices_sobrantes]

# Eliminar las imagenes escogidas
mascara = np.ones(len(X), dtype=bool)
mascara[indices_sobrantes] = False

X = X[mascara]
y = y[mascara]

In [6]:
shuffle_idx = rng.permutation(len(X))
X = X[shuffle_idx]
y = y[shuffle_idx]

## División de los datos

Los datos restantes se dividen en tres conjuntos:

- **Train:** utilizado para entrenar los modelos.
- **Validation:** utilizado para validar o ajustar los modelos.
- **Test:** utilizado para obtener la evaluación final sobre datos que no participaron en el entrenamiento.

Se utiliza `stratify` para mantener una proporción equivalente de gatos y perros en cada conjunto.

In [7]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

## Normalización de los píxeles

Cada dibujo contiene 784 valores correspondientes a una imagen de 28×28 píxeles. Los valores originales están entre 0 y 255, por lo que se dividen por 255 para llevarlos al intervalo **[0, 1]**.

La misma transformación se aplica a train, validation y test.

In [8]:
X_train = X_train.astype(np.float32) / 255.0
X_val   = X_val.astype(np.float32) / 255.0
X_test  = X_test.astype(np.float32) / 255.0

In [9]:
output_path = Path("datos/gatos_perros_dataset.npz")
individual_path = Path("datos/datos-prueba-individual.npz")
test_path = Path("datos/datos-test.npz")

# Dataset principal: train, validation y test
np.savez_compressed(
    output_path,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    X_test=X_test,
    y_test=y_test
)

# Datos reservados exclusivamente para la prueba individual
np.savez_compressed(
    individual_path,
    X_individual=X_sobrante,
    y_individual=y_sobrante
)

# Datos de test guardados también de forma independiente
np.savez_compressed(
    test_path,
    X_test=X_test,
    y_test=y_test
)

print("\nDatasets preparados:")
print(f"Train:      {X_train.shape[0]:,} dibujos")
print(f"Validation: {X_val.shape[0]:,} dibujos")
print(f"Test:       {X_test.shape[0]:,} dibujos")
print(f"Individual: {X_sobrante.shape[0]:,} dibujos")
print(f"\nDataset principal: {output_path}")
print(f"Datos de test:     {test_path}")
print(f"Prueba individual: {individual_path}")



Datasets preparados:
Train:      13,993 dibujos
Validation: 2,998 dibujos
Test:       2,999 dibujos
Individual: 10 dibujos

Dataset principal: datos\gatos_perros_dataset.npz
Datos de test:     datos\datos-test.npz
Prueba individual: datos\datos-prueba-individual.npz
